In [101]:
!pip install -q transformers datasets accelerate evaluate

In [102]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [103]:

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [104]:
print(train_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [105]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train_df["label"] = train_df["answer"].map(label_map)

In [106]:
def build_text(row):
    return (
        f"Question: {row['prompt']} "
        f"[SEP] A: {row['A']} "
        f"[SEP] B: {row['B']} "
        f"[SEP] C: {row['C']} "
        f"[SEP] D: {row['D']} "
        f"[SEP] E: {row['E']}"
    )

train_df["text"] = train_df.apply(build_text, axis=1)
test_df["text"] = test_df.apply(build_text, axis=1)

In [107]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

In [108]:
train_ds = Dataset.from_pandas(
    train_part[["text","label"]]
)

val_ds = Dataset.from_pandas(
    val_part[["text","label"]]
)

In [109]:
MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [110]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }

In [111]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [112]:
training_args = TrainingArguments(
    output_dir="./distilroberta_mcq",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    warmup_steps=100,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

In [113]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [114]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.540470,0.550000
2,No log,0.085203,0.995000
3,No log,0.004264,1.000000
4,No log,0.003059,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=452, training_loss=1.0387380819405074, metrics={'train_runtime': 132.4577, 'train_samples_per_second': 54.357, 'train_steps_per_second': 3.412, 'total_flos': 476908148736000.0, 'train_loss': 1.0387380819405074, 'epoch': 4.0})

In [115]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.0030588663648813963,
 'eval_accuracy': 1.0,
 'eval_runtime': 1.1199,
 'eval_samples_per_second': 178.588,
 'eval_steps_per_second': 11.608,
 'epoch': 4.0}

In [116]:
from datasets import Dataset

test_ds = Dataset.from_pandas(
    test_df[["text"]]
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [117]:
predictions = trainer.predict(test_ds)

logits = predictions.predictions

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

In [121]:
labels = np.array(["A","B","C","D","E"])

top3 = []

for row in probs:
    idx = np.argsort(row)[::-1][:3]
    top3.append(" ".join(labels[idx]))

In [122]:
submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = top3   # note capital P

submission = submission[["ID", "Prediction"]]

submission.head()

,ID,Prediction
0,1,A D C
1,2,B C E
2,3,B C D
3,4,E C D
4,5,C B E


In [123]:
submission.to_csv(
    "submission_distilroberta.csv",
    index=False
)

In [124]:
import os

print(os.listdir("/kaggle/working"))

['distilroberta_mcq', 'mpnet_mcq', 'submission.csv', '.virtual_documents', 'submission_distilroberta.csv']


In [125]:
import os

for f in os.listdir('/kaggle/working'):
    print(f)

distilroberta_mcq
mpnet_mcq
submission.csv
.virtual_documents
submission_distilroberta.csv


In [126]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [127]:
import os
print(os.listdir('/kaggle/working'))

['distilroberta_mcq', 'mpnet_mcq', 'submission.csv', '.virtual_documents', 'submission_distilroberta.csv']


In [128]:
df = pd.read_csv("submission.csv")
print(df.head())
print(df.columns.tolist())

   ID Prediction
0   1      A D C
1   2      B C E
2   3      B C D
3   4      E C D
4   5      C B E
['ID', 'Prediction']
